In [ ]:
rstfolder = '/home/jasp/sylvs/sentinel-2-may22/'

fformat = '.tif'

ocube = '/mnt/x/firebeds_tk1/lrifeat.zarr'

chunksize = (512, 512)

In [ ]:
import rioxarray
import xarray as xr
import numcodecs

from glass.pys.oss import lst_ff, fprop
from glass.rd.rst import rst_to_dset
from glass.prop.prj import get_epsg

In [ ]:
gtifs = lst_ff(rstfolder, file_format=fformat)

In [ ]:
# Carregar GeoTIFFs com chunks para tirar proveito da RAM disponível
datasets = rst_to_dset(gtifs, chunks=chunksize, api='rio')

In [ ]:
type(datasets[0])

In [ ]:
def check_compatibility(datasets, crs_list):
    """Verifica se todos os datasets têm mesmo shape e CRS."""
    shapes   = [ds.rio.shape for ds in datasets]
    if len(set(shapes)) > 1:
        raise ValueError(f"Shapes diferentes entre os GeoTIFFs: {shapes}")
    if len(set(crs_list)) > 1:
        raise ValueError(f"Sistemas de coordenadas diferentes: {crs_list}")

In [ ]:
# Carregar GeoTIFFs com chunks para tirar proveito da RAM disponível
datasets = rst_to_dset(gtifs, chunks=chunksize, api='rio')

check_compatibility(datasets, [get_epsg(r) for r in gtifs])

# Empilhar as features na dimensão 'feature'
data_cube = xr.concat(datasets, dim='feature')

# Nomear a dimensão 'feature' conforme os ficheiros
fnames = [fprop(rst, 'fn') for rst in gtifs]
data_cube = data_cube.assign_coords(feature=fnames)

In [ ]:
if not data_cube.name:
    data_cube.name = fprop(ocube, 'fn')

In [ ]:
print("Tipo:", type(data_cube))
print("Nome lógico:", data_cube.name)
print("Nome interno:", data_cube.feature.name)
print("Tem spatial_ref?", "spatial_ref" in data_cube.coords)

In [ ]:
print(data_cube.name)

In [ ]:
# Save to file
compressor = numcodecs.Zlib(level=5)
data_cube.to_zarr(ocube, encoding={
    data_cube.name: {'compressor': compressor, 'chunks': (1, chunksize[0], chunksize[1])}
})